In [ ]:
!pip install pydrive tqdm -U

In [ ]:
import json

# Create a Desktop OAuth Client ID @ https://console.cloud.google.com/apis/credentials
client_id = ''
client_secret = ''

secrets_dict = {
    "installed": {
        "client_id": client_id,
        "client_secret": client_secret,
        "auth_uri": "https://accounts.google.com/o/oauth2/auth",
        "token_uri": "https://oauth2.googleapis.com/token",
        "redirect_uris": ["urn:ietf:wg:oauth:2.0:oob", "http://localhost"]
    }
}

with open("client_secrets.json", "w") as f:
    json.dump(secrets_dict, f)

In [ ]:
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive

gauth = GoogleAuth()
gauth.settings['get_refresh_token'] = True
gauth.LoadClientConfigFile("client_secrets.json")
gauth.CommandLineAuth()
drive = GoogleDrive(gauth)

In [ ]:
import os
import time
from datetime import datetime
from tqdm import tqdm  # For a progress bar

shared_drive_id = ''
local_folder = ''
timestamp = datetime.now().strftime('%d-%b-%y %H:%M:%S')

def get_all_files_and_total_size(base_path):
    files = []
    total_size = 0
    for dirpath, _, filenames in os.walk(base_path):
        for fname in filenames:
            path = os.path.join(dirpath, fname)
            size = os.path.getsize(path)
            files.append((path, os.path.relpath(path, base_path)))
            total_size += size
    return files, total_size

folder_metadata = {
    'title': timestamp,
    'mimeType': 'application/vnd.google-apps.folder',
    'parents': [{'id': shared_drive_id}],
    'driveId': shared_drive_id
}
root_folder = drive.CreateFile(folder_metadata)
root_folder.Upload(param={'supportsAllDrives': True})
root_folder_id = root_folder['id']
print(f"✅ Created root folder in Shared Drive: {timestamp} (ID: {root_folder_id})")

def create_drive_folder_structure(local_base, rel_path, parent_id, folder_cache):
    parts = rel_path.split(os.sep)
    current_parent_id = parent_id

    for part in parts[:-1]:  # Only folder path
        folder_key = os.path.join(local_base, part)
        if folder_key not in folder_cache:
            folder_metadata = {
                'title': part,
                'mimeType': 'application/vnd.google-apps.folder',
                'parents': [{'id': current_parent_id}],
                'driveId': shared_drive_id
            }
            folder = drive.CreateFile(folder_metadata)
            folder.Upload(param={'supportsAllDrives': True})
            folder_cache[folder_key] = folder['id']
            print(f"📁 Created folder: {part} (ID: {folder['id']})")
        current_parent_id = folder_cache[folder_key]

    return current_parent_id

def upload_all_with_progress(files, total_size, base_path, root_drive_id):
    uploaded_size = 0
    folder_cache = {}  # Cache for created folders
    start_time = time.time()

    pbar = tqdm(total=total_size, unit='B', unit_scale=True, desc="Uploading")

    for full_path, relative_path in files:
        try:
            parent_id = create_drive_folder_structure(base_path, relative_path, root_drive_id, folder_cache)
            file_name = os.path.basename(full_path)

            f = drive.CreateFile({
                'title': file_name,
                'parents': [{'id': parent_id}],
                'driveId': shared_drive_id
            })
            f.SetContentFile(full_path)
            f.Upload(param={'supportsAllDrives': True})

            file_size = os.path.getsize(full_path)
            uploaded_size += file_size
            pbar.update(file_size)

            elapsed = time.time() - start_time
            speed = uploaded_size / elapsed if elapsed > 0 else 0
            remaining = (total_size - uploaded_size) / speed if speed > 0 else 0

            pbar.set_postfix({
                'Uploaded': f"{uploaded_size / (1024**2):.2f} MB",
                'Remaining': f"{remaining:.1f} sec"
            })
        except Exception as e:
            print(f"❌ Error uploading {relative_path}: {e}")
    
    pbar.close()
    print(f"\n✅ All files uploaded! Total uploaded: {uploaded_size / (1024**2):.2f} MB")

# === Main Execution ===
files, total_size = get_all_files_and_total_size(local_folder)
print(f"📦 Total files: {len(files)}")
print(f"📏 Total size: {total_size / (1024**2):.2f} MB")

upload_all_with_progress(files, total_size, local_folder, root_folder_id)